# Hybrid Search, From the Ground Up
### Keyword search + semantic search, built and combined step by step

Type two things into any online store's search box and watch what happens.

Search **`WH-1000XM5`** — an exact model number — and you want *that specific product*, first result, no interpretation.

Search **`something to block airplane engine sound`** and you want noise-cancelling headphones — even though not one product listing contains the words "airplane", "block", or "silence".

These are two completely different questions, and it turns out **no single technique answers both well**:

| The query | What it needs | The tool |
|---|---|---|
| `WH-1000XM5` | match the exact words | **BM25** (keyword search) |
| `something to block airplane engine sound` | understand the *meaning* | **embeddings** (semantic search) |

**Hybrid search** is the answer to "why not both?" — run each technique, then merge their results into one ranking. This notebook builds the whole thing from nothing.

---

### What you'll build, and in what order

We follow the same path the field itself took — each idea exists to fix a flaw in the one before it:

1. **Keyword search** — why matching words is the natural first idea
2. **TF** — counting words, and why counting alone fails
3. **IDF** — how rare words carry the real signal
4. **Saturation** — why the 50th mention shouldn't count as much as the 1st
5. **Length normalisation** — why long documents shouldn't win by accident
6. **Sparse vectors** — what BM25 actually produces, and how a database stores it
7. **Dense vectors** — why we need embeddings, and how they differ
8. **Score normalisation** — why you can't just add the two scores together
9. **Hybrid search** — combining both, with a dial to balance them
10. **Pinecone** — storing and searching both vectors in a real database

> **How to read this notebook.** Every number you see is real — computed live from 751 actual products, not made up for the example. Parts 1 and 2 run with no setup at all. Parts 3 and 4 need `OPENAI_API_KEY` and `PINECONE_API_KEY`; their code is complete and correct, just not executed here.

---
# Part 1 · The data

Before any search technique, we need something to search. We'll use a catalog of **751 real products** — headphones, speakers, kitchen gadgets, tools, clothing — the kind of messy, mixed inventory a real store has.

Everything in this notebook is computed against this catalog, so the very first step is simply to load it and look at it.

In [1]:

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# pip install pandas numpy openai pinecone

import re
import math
import numpy as np
import pandas as pd
from collections import Counter

DATA_URL = "products.csv"

df = pd.read_csv(DATA_URL, sep="|")     # pipe-delimited, not comma
print("raw shape:", df.shape)
df.head(3)

raw shape: (751, 7)


,Id,Name,Description,Price,PriceCurrency,SupplyAbility,MinimumOrder
0,d2559c95-bd28-49d8-b53a-538c34a25bcb,Saucony Men's Kinvara 13 Running Shoe,"When it comes to lightweight speed, nothing cr...",600.93,USD,396,574
1,c0717df8-fd27-4355-9c06-a8ad1ebb3e95,Accutire MS-4021B Digital Tire Pressure Gauge ...,About this item Heavy duty construction and ru...,476.98,USD,288,993
2,29ea5156-8582-4345-8672-65a8f4c1e30c,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...,This extraordinary fusion is designed to nouri...,216.74,EUR,640,707


### Preparing the text

Raw data is never quite ready to use. Two small fixes turn this catalog into something searchable:

- **Fill in missing descriptions.** A couple of products have no description at all (an empty cell). Left alone, that empty value would crash the word-splitting step later, so we replace it with an empty string.
- **Decide what "the document" actually is.** A shopper's words could match a product's *name* or its *description*, so we glue them into a single `text` field. That combined field is what we'll search.

> **Why This Matters.** Notice what we're *not* putting into `text`: price, stock level, currency. Those are facts you **filter** on ("under $500"), not words you **search** for. Keeping searchable text and filterable facts separate is a decision that pays off all the way through to Part 4.

In [5]:
df["Description"] = df["Description"].fillna("")
df["text"] = df["Name"] + ": " + df["Description"]

print("empty descriptions:", (df["Description"] == "").sum())
print("products:", len(df))
df[["Id", "text"]].head(3)

empty descriptions: 2
products: 751


,Id,text
0,d2559c95-bd28-49d8-b53a-538c34a25bcb,Saucony Men's Kinvara 13 Running Shoe: When it...
1,c0717df8-fd27-4355-9c06-a8ad1ebb3e95,Accutire MS-4021B Digital Tire Pressure Gauge ...
2,29ea5156-8582-4345-8672-65a8f4c1e30c,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...


---
# Part 2 · Building BM25 from scratch

**BM25** is the keyword-search algorithm that has quietly run search boxes for thirty years — it's the default in Elasticsearch, OpenSearch, and Lucene. Despite the intimidating name (it stands for "Best Match, 25th attempt"), it's built entirely from ideas you already have an instinct for.

We're going to build it ourselves, one idea at a time, so that by the end nothing about it is mysterious. Here's the plan, and crucially, *why each step exists*:

| Step | The idea | The problem it fixes |
|---|---|---|
| 2.1 Tokenization | split text into words | (the starting point) |
| 2.2 Term Frequency | count the words | — |
| 2.3 Inverse Document Frequency | weight rare words higher | counting treats "the" like "headphones" |
| 2.4 Scoring | saturation + length correction | repetition and length game the score |
| 2.5 Sparse vector | store it in a database | scores need to become searchable data |

> **Think About It.** As you go, notice that we never *design* BM25. We keep building the simplest thing that could work, watching it break on a concrete example, and fixing exactly that break. The famous formula is just the accumulated repairs.

## 2.1 · Tokenization — turning text into words

A search engine can't compare sentences directly; it compares **words**. So the very first job is to chop each product's text into a clean list of words. This is called **tokenization**, and each word is a **token**.

Our rule is deliberately simple: lowercase everything (so `"Wireless"` and `"wireless"` count as the same word), then keep only runs of letters and digits (throwing away punctuation, spaces, and symbols).

> **Key Takeaway.** Tokenization also happens to the *query*. When a shopper types `"wireless headphones"`, BM25 sees two **separate, independent** words — it will score `wireless` and `headphones` on their own and add the results. It has no idea they form a phrase, or that they usually go together. Every technique in this notebook inherits that "bag of independent words" view.

In [3]:
def tokenize(text):
    """Turn a piece of text into a clean list of lowercase words.

    re.findall(r"[a-z0-9]+", ...) keeps only runs of letters and digits,
    so "Sony WH-1000XM5!" becomes ["sony", "wh", "1000xm5"] - punctuation
    and spacing are dropped, and everything is lowercased first.
    """
    return re.findall(r"[a-z0-9]+", text.lower())

# Do this once for every product, up front. corpus_tokens[i] is the word list
# for product i, and we'll reuse it constantly - re-tokenizing on every search
# would be wasteful.
corpus_tokens = [tokenize(t) for t in df["text"]]

# See it on one real product, so "tokenization" stops being abstract:
example = df.iloc[13]["text"]
print("RAW TEXT:")
print(" ", example[:110], "...")
print()
print("TOKENS:")
print(" ", tokenize(example)[:16], "...")
print()
print("total documents tokenized:", len(corpus_tokens))

RAW TEXT:
  Sony WH-1000XM5: Sony WH-1000XM5 headphones provide industry-leading noise cancellation, superior sound qualit ...

TOKENS:
  ['sony', 'wh', '1000xm5', 'sony', 'wh', '1000xm5', 'headphones', 'provide', 'industry', 'leading', 'noise', 'cancellation', 'superior', 'sound', 'quality', 'and'] ...

total documents tokenized: 751


## 2.2 · Term Frequency (TF) — how often does a word appear?

So far we've only split documents into words. The next natural question is:

> *How important is each word inside a document?*

The simplest possible answer: **count how many times it appears.** A product whose description says "headphones" five times is probably more about headphones than one that says it once. This count is called **Term Frequency**, or **TF**.

At this stage we're doing nothing clever — just counting. We'll spend the next three sections discovering why counting *alone* isn't enough and fixing it. We also grab two length measurements here (each document's length, and the average), because BM25 needs them in §2.4.

In [4]:
# --- Length statistics, needed later in 2.4 for the length-correction step ---
doc_lengths = np.array([len(toks) for toks in corpus_tokens])
avgdl = doc_lengths.mean()          # avgdl = "average document length"

print(f"shortest document : {doc_lengths.min()} tokens")
print(f"longest  document : {doc_lengths.max()} tokens")
print(f"average  (avgdl)  : {avgdl:.2f} tokens")

# --- Term Frequency for one example document ---
# Counter(list_of_words) tallies how many times each word occurs. That tally
# IS the term frequency for this document.
DOC_ID = 13
doc_tokens = corpus_tokens[DOC_ID]
tf_counts = Counter(doc_tokens)

print(f"\nDocument {DOC_ID}: {df.iloc[DOC_ID]['Name']}")
print(f"  {len(doc_tokens)} tokens, {len(tf_counts)} distinct words")

# The most common words in this one product. Notice the winners are boring:
# "sony", "wh", "1000xm5" tie with filler like "and" - a hint that raw counts
# alone can't tell useful words from useless ones. Section 2.3 fixes exactly that.
pd.DataFrame(tf_counts.most_common(8), columns=["term", "tf"])

shortest document : 7 tokens
longest  document : 241 tokens
average  (avgdl)  : 76.76 tokens

Document 13: Sony WH-1000XM5
  26 tokens, 22 distinct words


,term,tf
0,sony,2
1,wh,2
2,1000xm5,2
3,and,2
4,headphones,1
5,provide,1
6,industry,1
7,leading,1


## 2.3 · Inverse Document Frequency (IDF) — which words actually matter?

Counting has a blind spot, and it's a big one. Let's find it with a concrete search rather than a definition.

Suppose someone searches for **`the wireless headphones`** — three words. Plain counting (§2.2) would treat all three as equally important. But are they? Let's ask the data how common each one is.

In [5]:
# --- Document frequency: in how many DIFFERENT products does each word appear? ---
# This is NOT the total number of occurrences. BM25 only cares whether a
# document contains the word at least once, because a word that shows up in
# almost every product (like "the") can't help you tell products apart.
doc_freq = Counter()
for toks in corpus_tokens:
    doc_freq.update(set(toks))        # set() collapses repeats -> count each doc once

N = len(corpus_tokens)                # total number of products

for term in ["the", "wireless", "headphones"]:
    n = doc_freq[term]
    print(f'"{term:11}" appears in {n:>4} of {N} products  ({100*n/N:5.1f}% of the catalog)')

"the        " appears in  582 of 751 products  ( 77.5% of the catalog)
"wireless   " appears in   22 of 751 products  (  2.9% of the catalog)
"headphones " appears in   11 of 751 products  (  1.5% of the catalog)


**Look at what each word actually did for the search.**

Matching on `"the"` leaves you with 582 products — 77% of the catalog. It narrowed nothing; you may as well not have typed it.

Matching on `"headphones"` leaves you with 11. *That* word did the work.

So a word is useful to a search in proportion to how **rare** it is. Common words are noise. This is the idea IDF captures, and now we just need a number for it.

### Attempt 1 — the simplest thing that could work

If rare = useful, then divide the total number of documents by how many contain the word:

$$\text{usefulness}(t) = \frac{N}{n(t)}$$

A word in 1 product out of 751 scores 751. A word in every product scores 1. Let's see it.

In [6]:
# Attempt 1: usefulness = N / n(t)
#   N    = total products (751)
#   n(t) = how many contain the word
# A word in 1 product scores 751; a word in every product scores ~1.
demo_terms = ["and", "the", "for", "wireless", "bluetooth", "headphones", "quietcomfort", "1000xm5"]

pd.DataFrame([
    {"term": t, "in_n_docs": doc_freq[t], "N / n(t)": round(N / doc_freq[t], 1)}
    for t in demo_terms
]).sort_values("N / n(t)")

,term,in_n_docs,N / n(t)
0,and,637,1.2
1,the,582,1.3
2,for,535,1.4
3,wireless,22,34.1
4,bluetooth,14,53.6
5,headphones,11,68.3
6,quietcomfort,2,375.5
7,1000xm5,1,751.0


The ordering is exactly right — filler words at the bottom, the rare model number at the top. But there's a hidden problem, and it's worth seeing concretely because it's the reason we don't stop here.

**Remember that scores get *added up* across the words in a query.** So imagine the search `wireless bluetooth headphones` and two competing products:

| Product | Matches | Attempt-1 score |
|---|---|---|
| A real wireless bluetooth headphone | `wireless` (34) + `bluetooth` (54) + `headphones` (68) | **156** |
| A phone case whose blurb mentions one rare code | `1000xm5` (751) | **751** |

The phone case wins by nearly 5× — on the strength of **one** rare word — over a product that genuinely matches **all three** search terms. That's clearly wrong.

> **Why This Matters.** The raw `N / n(t)` numbers are too *spread out*: from ~1 up to 751. One lucky rare word can outweigh several relevant common ones and hijack the whole ranking. We need to keep the ordering (rare still beats common) while taming that runaway spread. That's the next cell.

### Attempt 2 — squash the range with a logarithm

We want to keep the ordering but compress that runaway spread. A logarithm does exactly this: it pulls large numbers down hard and small numbers down gently.

$$idf(t) = \ln\!\left(\frac{N}{n(t)}\right)$$

In [7]:
# Attempt 2: take the logarithm of the ratio.
# A logarithm pulls large numbers down hard and small numbers down gently -
# exactly what we want: keep the ordering, kill the runaway spread.
pd.DataFrame([
    {
        "term": t,
        "in_n_docs": doc_freq[t],
        "N / n(t)": round(N / doc_freq[t], 1),
        "ln(N / n(t))": round(math.log(N / doc_freq[t]), 2),
    }
    for t in demo_terms
]).sort_values("ln(N / n(t))")

,term,in_n_docs,N / n(t),ln(N / n(t))
0,and,637,1.2,0.16
1,the,582,1.3,0.25
2,for,535,1.4,0.34
3,wireless,22,34.1,3.53
4,bluetooth,14,53.6,3.98
5,headphones,11,68.3,4.22
6,quietcomfort,2,375.5,5.93
7,1000xm5,1,751.0,6.62


Now compare the last two columns. The raw ratio ran 1.2 → 751 (a **600×** spread). After the logarithm it runs 0.16 → 6.62 — about **40×**.

Same ordering, sane range. A rare word is still worth much more than a common one, but no longer so much more that it drowns out everything else in the query.

**That is IDF.** The idea is complete.

### The production form

One detail before we move on. `ln(N / n(t))` breaks in two edge cases: a term appearing in *zero* documents divides by zero, and a term appearing in *every* document scores exactly 0, contributing nothing at all.

Real search engines add small constants to smooth both away:

$$idf(t) = \ln\!\left(1 + \frac{N - n(t) + 0.5}{n(t) + 0.5}\right)$$

It looks fussier, but it is the same idea with the sharp corners filed off — and the numbers barely move, which is the point of the comparison below.

In [8]:
def idf(term):
    """How much weight a word deserves, based on how rare it is.
    Higher = rarer = more useful for telling products apart.

    This is the production form. The (N - n + 0.5)/(n + 0.5) and the
    outer 1 + ... exist only to keep the number well-behaved in the two
    edge cases from the markdown above - the shape is still just "log of
    a rareness ratio", i.e. Attempt 2.
    """
    n = doc_freq.get(term, 0)
    return math.log(1 + (N - n + 0.5) / (n + 0.5))

# Give every distinct word a fixed integer id (its "slot number"). We'll need
# this in 2.5 when a document becomes a vector - each word maps to one dimension.
vocabulary = sorted(doc_freq)
term_to_id = {term: i for i, term in enumerate(vocabulary)}

print(f"corpus: {N} documents, {len(vocabulary):,} distinct terms\n")

# Compare our simple Attempt-2 formula against the production one, side by side.
# The point of this table is that they barely differ - the smoothing is cosmetic.
pd.DataFrame([
    {
        "term": t,
        "in_n_docs": doc_freq[t],
        "simple: ln(N/n)": round(math.log(N / doc_freq[t]), 2),
        "production idf": round(idf(t), 2),
    }
    for t in demo_terms
]).sort_values("production idf")

corpus: 751 documents, 8,468 distinct terms



,term,in_n_docs,simple: ln(N/n),production idf
0,and,637,0.16,0.17
1,the,582,0.25,0.26
2,for,535,0.34,0.34
3,wireless,22,3.53,3.51
4,bluetooth,14,3.98,3.95
5,headphones,11,4.22,4.18
6,quietcomfort,2,5.93,5.71
7,1000xm5,1,6.62,6.22


The two columns are nearly identical — 6.62 vs 6.22 at the top, 0.16 vs 0.17 at the bottom. The smoothing changes almost nothing in normal cases; it exists only to stop the edge cases misbehaving. **Use the production version, but the idea you should carry around is Attempt 2.**

### What this buys us

`"1000xm5"` scores **6.22**. `"the"` scores **0.26**. So when someone searches `"the 1000xm5"`, BM25 weights the model number about **24×** more heavily than the word `"the"`.

That single ratio is the difference between returning one exact product and returning three-quarters of the catalog.

## 2.4 · BM25 scoring

We have two ingredients now: **how often** a word appears (§2.2) and **how rare** it is (§2.3). Multiply them together and you almost have a working search engine.

Almost. Two things still go wrong, and BM25 is really just those two repairs. We'll take them one at a time.

### Problem 1 — repetition pays too well

With plain counting, a word appearing 50 times scores 50× a word appearing once.

So imagine a junk listing that repeats `"headphones"` fifty times in its description. It would beat the real Sony product page, which says it two or three times like a normal piece of writing. This isn't hypothetical — people spammed search engines this way for years, and the technique has a name: **keyword stuffing**.

**The repair:** let repetition still help, but pay less each time. The second mention should add less than the first, the tenth less than the second, and past a point extra mentions should barely register.

The knob controlling how fast it flattens is called **k₁**, conventionally 1.5.

In [9]:
k1 = 1.5   # the "saturation" dial. Higher = repetition keeps mattering longer.

def saturate(term_count):
    """Turn a raw count into a with-diminishing-returns credit.
    The 1st mention gives full credit; each extra mention gives a little less;
    the total can never exceed k1 + 1, no matter how many times the word repeats.
    """
    return term_count * (k1 + 1) / (term_count + k1)

print("How much credit does the Nth mention actually add?\n")
print(f"{'mentions':>9}{'plain count':>14}{'with saturation':>18}")
for tf in [1, 2, 3, 5, 10, 20, 50]:
    print(f"{tf:>9}{tf:>14}{saturate(tf):>18.3f}")
print(f"\nno matter how many times you repeat a word, it can never exceed k1 + 1 = {k1 + 1}")

How much credit does the Nth mention actually add?

 mentions   plain count   with saturation
        1             1             1.000
        2             2             1.429
        3             3             1.667
        5             5             1.923
       10            10             2.174
       20            20             2.326
       50            50             2.427

no matter how many times you repeat a word, it can never exceed k1 + 1 = 2.5


Compare the two right-hand columns. Plain counting runs away to 50; saturation climbs to 2.43 and stops.

Going from 1 mention to 5 nearly doubles the credit (1.00 → 1.92) — that's a real signal, and it's rewarded. Going from 20 to 50, which is more than double the mentions, moves the score by about 4%. **The spam listing gains almost nothing over an honest one.**

### Problem 2 — long documents win by accident

The second problem is subtler. Our product descriptions are wildly different lengths.

In [10]:
# How different ARE our document lengths? If they're all similar, this whole
# step barely matters. If they vary wildly, ignoring length would badly skew
# results toward the long ones. Let's check.
doc_lengths = np.array([len(toks) for toks in corpus_tokens])
avgdl = doc_lengths.mean()

print(f"shortest product listing : {doc_lengths.min():>3} tokens")
print(f"longest  product listing : {doc_lengths.max():>3} tokens")
print(f"average  (avgdl)         : {avgdl:.1f} tokens")
print(f"\nThe longest listing has {doc_lengths.max()/doc_lengths.min():.0f}x more words than the shortest -")
print("so it has far more chances to match a query word purely by being long.")

shortest product listing :   7 tokens
longest  product listing : 241 tokens
average  (avgdl)         : 76.8 tokens

The longest listing has 34x more words than the shortest -
so it has far more chances to match a query word purely by being long.


A 241-word listing isn't more relevant than a 7-word one — it's just wordier. But it collects matches simply by having more words, and without a correction it will drift to the top of every search.

**The repair:** divide a document's score down in proportion to how long it is, measured against the corpus average. A listing of exactly average length is left alone; longer ones get penalised, shorter ones get a small boost.

The knob controlling how aggressively is called **b**, conventionally 0.75. (`b = 0` means no correction at all, `b = 1` means full correction.)

In [11]:
b = 0.75   # the "length correction" dial. b=0 ignores length; b=1 corrects fully.

def length_penalty(doc_len):
    """A multiplier based on how long a document is vs. the corpus average.
    > 1  -> longer than average, its score gets divided down (penalised)
    ~ 1  -> about average, left alone
    < 1  -> shorter than average, gets a small boost
    """
    return 1 - b + b * doc_len / avgdl

print(f"corpus average is {avgdl:.0f} tokens\n")
for length in [7, 25, 77, 150, 241]:
    factor = length_penalty(length)
    if abs(factor - 1) < 0.02:
        verdict = "about average - left alone"
    elif factor > 1:
        verdict = "penalised (longer than average)"
    else:
        verdict = "rewarded (shorter than average)"
    print(f"{length:>4} tokens -> factor {factor:.2f}   {verdict}")

corpus average is 77 tokens

   7 tokens -> factor 0.32   rewarded (shorter than average)
  25 tokens -> factor 0.49   rewarded (shorter than average)
  77 tokens -> factor 1.00   about average - left alone
 150 tokens -> factor 1.72   penalised (longer than average)
 241 tokens -> factor 2.60   penalised (longer than average)


### Putting the two repairs together

Now combine everything. For one query term `t` in one document `d`:

$$score(t,d) = \underbrace{idf(t)}_{\substack{\text{how rare} \\ \text{§2.3}}} \times \underbrace{\frac{tf \cdot (k_1+1)}{tf + k_1 \cdot \text{length penalty}}}_{\substack{\text{how often, saturated} \\ \text{and length-corrected}}}$$

Read it in plain English: **how rare the word is, times how often it appears (with diminishing returns), divided down if the document is long.** Sum that across every word in the query and you have BM25 — nothing else.

Everything in that formula is something we just built. Nothing new is being introduced.

In [12]:
def tf_component(term_count, doc_len):
    """The 'how often, done right' half of BM25 for one word in one document:
    saturating term frequency (2.4 problem 1), corrected for length (2.4 problem 2).

    Important: this depends ONLY on the document itself - the word's count and
    the document's length. No corpus-wide statistics appear here. Remember that;
    it's the key to how the sparse vector is stored in 2.5.
    """
    return term_count * (k1 + 1) / (term_count + k1 * length_penalty(doc_len))

def bm25_score(query, doc_id):
    """Full BM25 score of one document against one query.
    For each query word the document actually contains, multiply:
        idf(word)             -> how rare/useful the word is   (corpus fact, 2.3)
        tf_component(word,doc) -> how strongly this doc uses it (doc fact, 2.4)
    then add those up across all the query's words.
    """
    doc = corpus_tokens[doc_id]
    counts = Counter(doc)
    return sum(idf(t) * tf_component(counts[t], len(doc))
               for t in tokenize(query) if t in counts)

def bm25_search(query, k=5):
    """Score every product against the query and return the top k."""
    scores = np.array([bm25_score(query, i) for i in range(N)])
    top = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({
        "score": [round(scores[i], 3) for i in top],
        "product": [df.iloc[i]["Name"][:58] for i in top],
    })

# The moment of truth - a real keyword search, built entirely by hand:
bm25_search("wireless noise cancelling headphones")

,score,product
0,24.021,Bose 700 Noise Cancelling Headphones
1,23.335,Bose Noise Cancelling Headphones 700
2,19.252,Sony WH-1000XM4 Noise Cancelling Headphones
3,18.991,Sennheiser Momentum 3 Wireless Headphones
4,18.849,Sony WH-1000XM4 Noise Cancelling Headphones


That's a genuine, working keyword search engine — and we built every piece of it ourselves, no library involved. For an exact query like this, it's excellent.

But now we deliberately break it, because the way it fails is the entire reason the rest of this notebook exists. Watch what happens when a shopper describes what they *want* instead of naming it:

In [13]:
bm25_search("something to block airplane engine sound")

,score,product
0,10.104,Dorman 603-026 Front Engine Coolant Reservoir ...
1,6.550,WallFlower Women's Luscious Curvy Bootcut Mid-...
2,6.300,AsterOutdoor Sun Shade Sail Rectangle 12' x 16...
3,5.617,Bose QuietComfort 45
4,5.529,"Rotring 600 Ballpoint Pen, Medium Point, Blue ..."


A shopper asked how to block airplane engine noise — and got a **car engine coolant reservoir**, women's jeans, and a UV-blocking sun shade.

Nothing is broken. This is BM25 working *exactly as designed*. The word `engine` is rare in a product catalog, so §2.3 gives it heavy weight, and the coolant reservoir is genuinely the best *word* match. `block` matched the sun shade.

The real trouble: **not one product listing contains the words `airplane`, `silence`, or `noise-cancelling-for-a-flight`**. The shopper described a *problem*; the catalog describes *products*, using different words for the same idea. With no shared vocabulary, there is simply nothing for keyword search to grab onto.

> **Key Takeaway.** BM25 matches *words*, not *meaning*. It will never connect "block airplane engine sound" to "active noise cancellation" because the two share no words — no matter how obvious the connection is to a human. This exact gap is what **dense embeddings** (Part 3) are built to close. Hold onto this airplane query; we'll bring it back.

## 2.5 · Sparse vectors and dense vectors

We have a working keyword search. To put it in a database, those scores have to become **vectors** — lists of numbers. There are two kinds, and the difference between them is the last concept you need before building the real thing.

### A sparse vector = one slot per word

Picture the whole catalog as a giant grid:

- **each row is a product**
- **each column is a word** from the vocabulary
- **each cell holds that word's weight in that product** — and `0` if the product doesn't contain the word at all

Let's build a small corner of that grid and look at it.

In [14]:
# A small corner of the "documents x words" grid, built from real data.
# Rows = 4 products. Columns = 8 words chosen from the vocabulary.
# Each cell = that word's BM25 weight in that product (0 = word not present).
sample_rows = [13, 244, 102, 377]
sample_terms = ["1000xm5", "headphones", "bluetooth", "speaker",
                "wireless", "bass", "sony", "jeans"]

grid = []
for i in sample_rows:
    counts = Counter(corpus_tokens[i])
    doc_len = len(corpus_tokens[i])
    grid.append([
        round(tf_component(counts[t], doc_len), 2) if counts[t] > 0 else 0
        for t in sample_terms
    ])

matrix = pd.DataFrame(grid,
                      columns=sample_terms,
                      index=[df.iloc[i]["Name"][:28] for i in sample_rows])

filled = (matrix.values != 0).sum()
total = matrix.size
print(f"this corner is {matrix.shape[0]} products x {matrix.shape[1]} words = {total} cells")
print(f"  filled : {filled}")
print(f"  zeros  : {total - filled}\n")
matrix

this corner is 4 products x 8 words = 32 cells
  filled : 9
  zeros  : 23



,1000xm5,headphones,bluetooth,speaker,wireless,bass,sony,jeans
Sony WH-1000XM5,1.81,1.42,0.00,0.00,0,0.00,1.81,0
Bose 700 Noise Cancelling He,0.00,1.82,0.00,0.00,0,0.00,0.00,0
JBL Charge 5 Bluetooth Speak,0.00,0.00,1.35,1.75,0,1.35,0.00,0
Sony WH-1000XM4 Noise Cancel,0.00,1.39,0.00,0.00,0,0.00,1.79,0


Read it as a table and the structure is obvious:

- **Sony WH-1000XM5** has a weight under `1000xm5`, `headphones` and `sony` — and `0` everywhere else.
- **JBL Charge 5** has weights under `bluetooth`, `speaker` and `bass` — and `0` under everything the headphones matched.
- **`jeans` is `0` for all four**, because none of these products is a pair of jeans. That column would only light up for a clothing product.

Even in this tiny 4×8 corner, most cells are already zero. Now scale it to the real thing:

| | |
|---|---|
| rows (products) | 751 |
| columns (vocabulary words) | 8,468 |
| **total cells** | **≈ 6.4 million** |
| cells that aren't zero | about 16,000 |
| **zeros** | **over 99%** |

> **Why This Matters.** That full 6.4-million-cell grid is the *mental picture*, not the storage. **It is never actually built.** Holding 6.4 million numbers to store 16,000 real ones would be absurd — so instead, each row is stored on its own as just the positions that are filled.

Here's one complete row — every non-zero cell for a single product:

In [15]:
# What the sparse vector for one product actually looks like.
# Each row is one NON-ZERO slot - every other slot in the vector is 0.
sparse_df = pd.DataFrame([
    {
        "slot": term_to_id[term],           # this word's fixed position in the vector
        "term": term,                       # what that slot MEANS
        "tf":   count,                      # how often the word is in this product
        "weight": round(tf_component(count, len(doc_tokens)), 4),   # the stored number
    }
    for term, count in Counter(doc_tokens).items()
]).sort_values("slot").reset_index(drop=True)

print(f"{df.iloc[DOC_ID]['Name']}")
print(f"  vocabulary   : {len(vocabulary):,} slots total")
print(f"  non-zero     : {len(sparse_df)} slots")
print(f"  zeros        : {100 * (1 - len(sparse_df)/len(vocabulary)):.1f}% of the vector\n")
sparse_df.head(8)

Sony WH-1000XM5


  vocabulary   : 8,468 slots total
  non-zero     : 22 slots
  zeros        : 99.7% of the vector



,slot,term,tf,weight
0,38,1000xm5,2,1.8142
1,328,30,1,1.4236
2,990,and,2,1.8142
3,1270,battery,1,1.4236
4,1666,calls,1,1.4236
5,1683,cancellation,1,1.4236
6,3394,for,1,1.4236
7,3851,headphones,1,1.4236


That's the same product's row from the grid, with the zeros left out — 22 filled cells out of 8,468.

So a sparse vector is stored as **two short lists**:

```
indices : [38,    990,   1270,  3851,  ...]   <- which columns are filled
values  : [1.8142, 1.8142, 1.4236, 1.4236, ...]   <- what's in them
```

> **Key point about the columns.** Column 38 must mean `"1000xm5"` for *every* product, forever. If the numbering shifted between documents, a query looking at column 38 would be comparing against a different word in each one. This is the sparse equivalent of the "same embedding model everywhere" rule on the dense side.

This `{indices, values}` pair is exactly the format Pinecone accepts — and it's why sparse vectors stay cheap even with a huge vocabulary.

### A dense vector = meaning, compressed

A **dense vector** is the opposite in every way. An embedding model reads the text and returns a fixed list of numbers — 1,536 of them for OpenAI's model — where *every* slot is filled and **no slot corresponds to any particular word**. The meaning is spread across all of them at once.

There's no grid to draw here, because there are no word-columns to label. That's precisely what lets it solve the airplane query from §2.4: "block airplane engine sound" and "active noise cancellation" land close together because they *mean* the same thing — with zero words in common.

| | **Sparse** (BM25) | **Dense** (embeddings) |
|---|---|---|
| Size | 8,468 slots, grows with vocabulary | 1,536 slots, fixed |
| How full | ~20 filled, rest zero | all 1,536 filled |
| A slot means | one specific word | nothing nameable |
| Made by | counting (Part 2) | a trained model |
| Finds | the exact words you typed | the meaning behind them |
| Blind to | meaning, synonyms | exact codes like `WH-1000XM5` |

> **Key Takeaway.** Their blind spots don't overlap. BM25 is blind exactly where embeddings see (meaning); embeddings are blind exactly where BM25 sees (exact rare tokens). Store both, search both — that's hybrid search.

---
# Part 3 · Encoding the catalog for real

We built BM25 by hand so nothing about it is mysterious. **In production you don't hand-roll it** — Pinecone ships a `BM25Encoder` that does exactly what we built (fit corpus stats, encode documents and queries into `{indices, values}`), tested and optimised.

So from here we use the library for sparse, and OpenAI for dense.

> **Heads up.** The remaining cells need `OPENAI_API_KEY` and `PINECONE_API_KEY` and were **not run here** (no network in this environment). The code is complete — add your keys and it runs as written.

In [9]:
import ssl, certifi, functools
ssl._create_default_https_context = functools.partial(
    ssl.create_default_context, cafile=certifi.where()
)

In [7]:
! pip install nltk


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [10]:
# pip install pinecone-text
from pinecone_text.sparse import BM25Encoder

# The library version of everything we built in Part 2.
# .fit() learns the corpus statistics - which words are rare (IDF) and the
# average document length - exactly the numbers we computed by hand.
bm25 = BM25Encoder()
bm25.fit(df["text"].tolist())

# Two different methods, matching the document/query split we saw in 2.4:
#   encode_documents -> the "how often, length-corrected" half  (stored)
#   encode_queries   -> the "how rare" half                      (used at search time)
sparse_vectors = [bm25.encode_documents(t) for t in df["text"]]

print(sparse_vectors)

print("sparse vectors built:", len(sparse_vectors))
print("example - non-zero slots:", len(sparse_vectors[0]["indices"]))
print("format:", list(sparse_vectors[0].keys()), "<- exactly what Pinecone wants")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/akellaprudhvi/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/akellaprudhvi/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


  0%|          | 0/751 [00:00<?, ?it/s]

[{'indices': [777682453, 632192512, 691409538, 1611310005, 3075297601, 4051061832, 303109060, 2542944140, 2333455156, 997512866, 695926169, 2123550945, 2710963736, 3928038441, 1975257448, 3098195391, 2257684172, 3177574277, 1520739637, 705362877, 4248033388, 2608702916, 254839692, 2136405083, 1477105254, 2361787631, 1833938074, 3471378517, 3981855590, 3831243209, 3076736765], 'values': [0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.700527082890996, 0.5390855582042373, 0.5390855582042373, 0.700527082890996, 0.5390855582042373, 0.5390855582042373, 0.700527082890996, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.5390855582042373, 0.53908555820

In [ ]:
import ssl, certifi, functools
ssl._create_default_https_context = functools.partial(
    ssl.create_default_context, cafile=certifi.where()
)

In [11]:
import os
from openai import OpenAI

oa = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

EMBED_MODEL = "text-embedding-3-small"   # returns 1,536 numbers per text
EMBED_DIM = 1536

def embed_batch(texts):
    """Turn a list of texts into dense vectors, one API call for the whole batch."""
    response = oa.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in response.data]

# Embed every product, 100 at a time.
dense_vectors = []
BATCH = 100
for start in range(0, len(df), BATCH):
    chunk = df["text"].iloc[start:start + BATCH].tolist()
    dense_vectors.extend(embed_batch(chunk))

print(len(dense_vectors), "dense vectors of", len(dense_vectors[0]), "dimensions")

751 dense vectors of 1536 dimensions


---
# Part 3.5 · One catch before we combine them

Every product now has two scores: **BM25** (keywords) and **dense** (meaning). We need one ranked list, so they have to be merged.

You can't just add them — they're on totally different scales. A top BM25 score is around **24** with no upper limit; a top cosine score is around **0.6** and can never exceed 1. Add them and BM25 swamps the result every time, winning on scale rather than relevance.

The fix is to rescale both to the same range first, then blend. **Pinecone does this for us** — we just choose how much each side counts, using a single dial called `alpha` (next section).

---
# Part 4 · Hybrid search in Pinecone

Everything so far has been on our laptop. Now we put both vectors into a real vector database and let it do the hybrid search for us.

Pinecone stores a **sparse vector and a dense vector on the same record**, and when you query, it blends them server-side and returns one ranked list — the production version of everything we built by hand in Part 3.5.

> **Why This Matters — one hard rule.** The index metric must be **`dotproduct`**. Hybrid sparse-dense search does *not* work with `cosine` or `euclidean`; Pinecone will reject the query. This is the single most common setup mistake, so we set it correctly from the start.

In [13]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
INDEX_NAME = "vidyasankalp-hybrid"

# Create the index once. Two things matter here:
#   dimension = the DENSE size (1536). Sparse vectors need no size declared -
#               Pinecone handles their {indices, values} form automatically.
#   metric    = "dotproduct" - REQUIRED for hybrid. cosine/euclidean won't work.
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(INDEX_NAME)

## 4.1 · Storing both vectors on every product

Each record carries **both** representations of the same product:

- `values` — the **dense** vector (meaning)
- `sparse_values` — the **sparse** vector (exact words)
- `metadata` — the filterable facts we set aside in Part 1

One product, both ways of finding it, plus the facts to filter on.

In [14]:
# Upload all products, 100 at a time. The key line is that each record carries
# BOTH vectors: the dense one (meaning) and the sparse one (exact words).
BATCH = 100
for start in range(0, len(df), BATCH):
    vectors = []
    for i in range(start, min(start + BATCH, len(df))):
        row = df.iloc[i]
        vectors.append({
            "id": row["Id"],
            "values": dense_vectors[i],          # dense  (OpenAI)
            "sparse_values": sparse_vectors[i],  # sparse (BM25Encoder)
            "metadata": {                        # facts to filter on
                "text": row["text"],
                "price": float(row["Price"]),
                "currency": row["PriceCurrency"],
            },
        })
    index.upsert(vectors=vectors)

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=751, metric='dotproduct', namespaces=1)


## 4.2 · The `alpha` dial

Pinecone has no `alpha` argument. You set the balance yourself by **scaling the two query vectors before sending them** — whatever weight you multiply into a vector is the weight it carries in the search.

**`alpha` is the weight on the dense (meaning) side:**

| alpha | Leans toward | Good for |
|---|---|---|
| `0.0` | sparse only — pure keywords | exact codes, model numbers, SKUs |
| **below 0.5** | more keywords | catalogs full of part numbers |
| `0.5` | even blend | a sensible default |
| **above 0.5** | more meaning | natural-language queries |
| `1.0` | dense only — pure meaning | descriptive, vague searches |

> **Watch the direction.** Higher alpha = *more semantic*, because alpha multiplies the dense vector and `(1 - alpha)` multiplies the sparse one. It's easy to remember backwards, and getting it wrong silently inverts your search behaviour.

In [15]:
def scale_hybrid(dense_vec, sparse_vec, alpha):
    """Apply the alpha balance by scaling each query vector before sending it.

    alpha is the weight on the DENSE (meaning) side:
        dense  values are multiplied by  alpha
        sparse values are multiplied by  (1 - alpha)

    Pinecone then dot-products each against its stored counterpart, so the
    scaling we do here decides how much each side counts in the final ranking.
    """
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1")
    scaled_sparse = {
        "indices": sparse_vec["indices"],
        "values":  [v * (1 - alpha) for v in sparse_vec["values"]],
    }
    scaled_dense = [v * alpha for v in dense_vec]
    return scaled_dense, scaled_sparse

## 4.3 · One query, both retrievers, end to end

This ties every part of the notebook together in a single function:

- the **sparse** side of the query comes from `bm25.encode_queries()` — the library version of the IDF weighting we built in Part 2
- the **dense** side comes from `embed_batch()` — the OpenAI embedding from Part 3
- `scale_hybrid()` applies the `alpha` balance from §4.2
- Pinecone dot-products both, fuses them, and hands back one ranked list

In [16]:
def hybrid_search(query, alpha=0.5, top_k=5, metadata_filter=None):
    """Run one hybrid search, combining both retrievers.

    Steps:
      1. dense side : embed the query with OpenAI
      2. sparse side: encode the query with BM25Encoder
      3. balance    : scale both by alpha (section 4.2)
      4. search     : Pinecone dot-products both and fuses the results
    """
    dense_q = embed_batch([query])[0]        # meaning half
    sparse_q = bm25.encode_queries(query)    # keyword half

    dense_q, sparse_q = scale_hybrid(dense_q, sparse_q, alpha)

    result = index.query(
        vector=dense_q,
        sparse_vector=sparse_q,
        top_k=top_k,
        include_metadata=True,
        filter=metadata_filter,
    )

    return pd.DataFrame([
        {"score": round(m["score"], 4), "product": m["metadata"]["text"][:58]}
        for m in result["matches"]
    ])

**What you'll see when you run this with real keys** — and it's the payoff for the whole notebook:

- **`something to block airplane engine sound`** — *pure sparse* returns the coolant reservoir from §2.4 (the failure we started with). *Pure dense* returns noise-cancelling headphones. *Hybrid* keeps the headphones. Meaning rescued the query.
- **`WH-1000XM5`** — *pure sparse* nails the exact model decisively. *Pure dense* finds it but ranks the older XM4 almost equally, because to the model the two listings mean nearly the same thing. *Hybrid* keeps the exact match on top. Keywords rescued the query.

> **Key Takeaway.** Same system, opposite queries, both answered well — because the two retrievers cover each other's blind spots. That is the entire reason hybrid search exists.

In [17]:
# The two queries that defined this whole notebook, run at three settings of the
# dial. Watch each query do best at a different alpha - that's hybrid earning its keep.
for query in ["something to block airplane engine sound", "WH-1000XM5"]:
    print(f"\n=== {query} ===")
    for alpha, label in [(0.0, "pure sparse"), (0.5, "hybrid"), (1.0, "pure dense")]:
        print(f"\n  alpha={alpha}  ({label})")
        print(hybrid_search(query, alpha=alpha, top_k=3).to_string(index=False))


=== something to block airplane engine sound ===

  alpha=0.0  (pure sparse)
 score                                                    product
0.1426 World Traveler 22 Inch Duffle Bag, Pink Trim Zebra, One Si
0.1346 6 Pack 4"x6" Pale Pink Rubber Stamp Carving Blocks Rubber 
0.1251 PIT66 Dash Board Bezel Replace Compatible with Blazer Jimm

  alpha=0.5  (hybrid)
 score                                                    product
0.1805 Sony WH-1000XM4 Noise Cancelling Headphones: The Sony WH-1
0.1772 Sony WH-1000XM4 Noise Cancelling Headphones: The Sony WH-1
0.1733 Bose 700 Noise Cancelling Headphones: Bose 700 Noise Cance

  alpha=1.0  (pure dense)
 score                                                    product
0.2812 Heatshield Products 175105 Heatshield Armor 1/2" Thick x 1
0.2744 Sony WH-1000XM4 Noise Cancelling Headphones: The Sony WH-1
0.2669 Sony WH-1000XM4 Noise Cancelling Headphones: The Sony WH-1

=== WH-1000XM5 ===

  alpha=0.0  (pure sparse)
 score                          

## 4.4 · Adding metadata filters

Hybrid search and filtering compose cleanly. The two vectors decide *what's relevant*; the metadata filter decides *what's allowed*. Here we ask for premium wireless headphones, but only USD products under $500.

> **Think About It.** This is why we split searchable text from filterable facts all the way back in Part 1. The vectors never had to "know" about price — that's a hard rule applied afterward, and it's exactly how you'd enforce things like in-stock-only, or a single customer's data in a multi-tenant system.

In [18]:
# Vectors find what's relevant; the filter restricts to what's allowed.
# Only USD products priced under $500 are eligible to be returned.
hybrid_search(
    "premium wireless headphones",
    alpha=0.5,
    top_k=5,
    metadata_filter={"currency": {"$eq": "USD"}, "price": {"$lt": 500}},
)

,score,product
0,0.6214,Bose QuietComfort 35 II Wireless Headphones: T...
1,0.6200,Sennheiser Momentum 3 Wireless Headphones: The...
2,0.5520,Bose QuietComfort 45: Bose QuietComfort 45 hea...
3,0.4587,Bose Noise Cancelling Headphones 700: The Bose...
4,0.3967,Apple AirPods Pro 2nd Generation: Apple AirPod...


---
# Recap — the whole journey

**Part 1 — the data.** 751 real products. `Name + Description` is the searchable text; price and currency stay as filterable facts.

**Part 2 — BM25, built by hand.** Each idea earned by fixing the flaw before it:
- **tokenize** → split into words
- **TF** → count them (but counting treats "the" like "headphones")
- **IDF** → weight rare words higher (fixes that)
- **saturation** → stop repetition dominating the score
- **length normalisation** → stop long documents winning by accident

**§2.5 — two kinds of vector.** Sparse = one slot per word, mostly zeros, stored as `{indices, values}`. Dense = 1,536 filled slots carrying meaning. Complementary blind spots.

**Part 3 — production encoding.** `BM25Encoder` for sparse, OpenAI for dense. We built BM25 by hand to understand it; the library does it for real.

**Part 3.5 — combining scores.** The two live on different scales, so you can't just add them — rescale both to 0–1, then blend with `alpha`.

**Part 4 — Pinecone.** Both vectors on one record, `metric="dotproduct"`, `alpha` applied by scaling the query vectors. One query, both retrievers, one ranked answer.

> **The one rule that never changes.** Query and documents must be encoded the same way — same embedding model for dense, same fitted `BM25Encoder` for sparse. Refit or swap one, and you must rebuild the index.

> **Where to go next.** Tune `alpha` on your own queries; build a small evaluation set to measure recall instead of eyeballing results; look at Reciprocal Rank Fusion as a no-tuning alternative.